# Import, adattisztítás

In [ ]:
# Import

from fbref_module import *

url = "https://fbref.com/en/comps/882/schedule/Conference-League-Scores-and-Fixtures"
table_id = "sched_all"

ucl_data_raw = scrape(url, table_id)
print(ucl_data_raw.shape)

In [ ]:
# Clean data
import pandas as pd
import numpy as np
from datetime import datetime
import re

ucl_data_raw = ucl_data_raw[pd.notna(ucl_data_raw.Date) & (ucl_data_raw.Date != 'Date')]

def clean_team_name(name):
    # töröljük az országkódokat az elejéről vagy a végéről
    name = re.sub(r'\b[a-z]{2,3}\b', '', name)  # országkód törlése
    name = name.replace('.', '').strip()
    return re.sub(r'\s+', ' ', name)  # extra szóközök eltüntetése

ucl_data_raw['Home'] = ucl_data_raw['Home'].apply(clean_team_name)
ucl_data_raw['Away'] = ucl_data_raw['Away'].apply(clean_team_name)

ucl_data_raw['Goals_Home'] = ucl_data_raw['Score'].str.split('–').str[0]
ucl_data_raw['Goals_Away'] = ucl_data_raw['Score'].str.split('–').str[1]

ucl_data_raw.rename(columns={'xG': 'xG_Home', 'xG.1': 'xG_Away'}, inplace=True)


ucl_data_raw['Points_Home'] = np.where(ucl_data_raw.Goals_Home > ucl_data_raw.Goals_Away,
                                       3, np.where(ucl_data_raw.Goals_Home == ucl_data_raw.Goals_Away,
                                                   1, 0)
                                        )

ucl_data_raw['Points_Away'] = np.where(ucl_data_raw.Goals_Home < ucl_data_raw.Goals_Away,
                                       3, np.where(ucl_data_raw.Goals_Home == ucl_data_raw.Goals_Away,
                                                   1, 0)
                                        )


ucl_data_raw.Date = pd.to_datetime(ucl_data_raw.Date, format='%Y-%m-%d')
ucl_data_raw[['Goals_Home', 'Goals_Away', 
              'Points_Home', 'Points_Away',
              'xG_Home', 'xG_Away']] = ucl_data_raw[['Goals_Home', 'Goals_Away', 
                                                'Points_Home', 'Points_Away',
                                                'xG_Home', 'xG_Away']].astype(float)


mask = (ucl_data_raw['Date'] <= datetime.today())
ucl_data = ucl_data_raw.loc[mask, ['Date', 'Home', 'Away', 
                               'Goals_Home', 'Goals_Away', 
                               'xG_Home', 'xG_Away', 
                               'Points_Home', 'Points_Away']].copy()

print(f"\nMatches played: {len(ucl_data)}/{len(ucl_data_raw)}")
print(f"\nMatches with score: {len(ucl_data_raw[pd.notna(ucl_data_raw.Score)])}/{len(ucl_data)}")

display(ucl_data.head())

In [ ]:
# Pots

ucl_pot_dict = {
    # --- Pot 1 ---
    "Paris S-G": 1,
    "Real Madrid": 1,
    "Manchester City": 1,
    "Bayern Munich": 1,
    "Liverpool": 1,
    "Inter": 1,
    "Chelsea": 1,
    "Dortmund": 1,
    "Barcelona": 1,

    # --- Pot 2 ---
    "Arsenal": 2,
    "Leverkusen": 2,
    "Atlético Madrid": 2,
    "Benfica": 2,
    "Atalanta": 2,
    "Villarreal": 2,
    "Juventus": 2,
    "Eint Frankfurt": 2,
    "Club Brugge": 2,

    # --- Pot 3 ---
    "Tottenham": 3,
    "PSV Eindhoven": 3,
    "Ajax": 3,
    "Napoli": 3,
    "Sporting CP": 3,
    "Olympiacos": 3,
    "Slavia Prague": 3,
    "Bodø/Glimt": 3,
    "Marseille": 3,

    # --- Pot 4 ---
    "FC Copenhagen": 4,
    "Monaco": 4,
    "Galatasaray": 4,
    "Union SG": 4,
    "Qarabağ": 4,
    "Athletic Club": 4,
    "Newcastle Utd": 4,
    "Pafos FC": 4,
    "Qaırat Almaty": 4
}

uel_pot_dict = {
    # --- Pot 1 ---
    "Roma": 1,
    "Porto": 1,
    "Rangers": 1,
    "Feyenoord": 1,
    "Lille": 1,
    "Dinamo Zagreb": 1,
    "Betis": 1,
    "RB Salzburg": 1,
    "Aston Villa": 1,

    # --- Pot 2 ---
    "Fenerbahçe": 2,
    "Braga": 2,
    "Red Star": 2,  # Crvena Zvezda
    "Lyon": 2,
    "PAOK": 2,
    "Viktoria Plzeň": 2,
    "Ferencváros": 2,
    "Celtic": 2,
    "Maccabi Tel Aviv": 2,

    # --- Pot 3 ---
    "Young Boys": 3,
    "Basel": 3,
    "Midtjylland": 3,
    "Freiburg": 3,
    "Ludogorets": 3,
    "Nott' Forest": 3,
    "Sturm Graz": 3,
    "FCSB": 3,
    "Nice": 3,

    # --- Pot 4 ---
    "Bologna": 4,
    "Celta Vigo": 4,
    "Stuttgart": 4,
    "Panathinaikos": 4,
    "Malmö": 4,
    "Go Ahead Eag": 4,
    "Utrecht": 4,
    "Genk": 4,
    "Brann": 4
}

conf_pot_dict = {
    # --- Pot 1 ---
    "Fiorentina": 1,
    "AZ Alkmaar": 1,
    "Shakhtar": 1,  # Shakhtar Donetsk
    "Slovan Bratislava": 1,
    "Rapid Wien": 1,  # SK Rapid
    "Legia Warsaw": 1,  # Legia Warszawa

    # --- Pot 2 ---
    "Sparta Prague": 2,  # Sparta Praha
    "Dynamo Kyiv": 2,
    "Crystal Palace": 2,
    "Lech Poznań": 2,
    "Rayo Vallecano": 2,
    "Shamrock Rovers": 2,

    # --- Pot 3 ---
    "AC Omonia": 3,  # Omonoia
    "Mainz 05": 3,
    "Strasbourg": 3,
    "Jagiellonia": 3,  # Jagiellonia Białystok
    "NK Celje": 3,
    "Rijeka": 3,

    # --- Pot 4 ---
    "Zrinjski Mostar": 4,  # Zrinjski (BIH)
    "Red Imps": 4,  # Lincoln Red Imps
    "KuPS": 4,  # KuPS Kuopio
    "AEK Athens": 4,
    "Aberdeen": 4,
    "KF Drita": 4,  # Drita (KOS)

    # --- Pot 5 ---
    "Breiðablik": 5,
    "Sigma Olomouc": 5,
    "Samsunspor": 5,
    "Raków": 5,
    "AÉK Lárnaka": 5,  # AEK Larnaca
    "Shkëndija 79": 5,  # Shkëndija (MKD)

    # --- Pot 6 ---
    "Häcken": 6,
    "Lausanne-Sport": 6,
    "CS U Craiova": 6,  # Universitatea Craiova
    "Ħamrun Spartans FC": 6,
    "FC Noah": 6,
    "Shelbourne FC": 6
}


pot_dict = conf_pot_dict.copy()

ucl_data['Pot_Home'] = ucl_data['Home'].map(pot_dict)
ucl_data['Pot_Away'] = ucl_data['Away'].map(pot_dict)

# ellenőrzés, hogy nincs-e hiányzó érték
missing_teams = set(ucl_data['Home']).union(ucl_data['Away']) - set(pot_dict.keys())
print("Hiányzó a pot dictionary-ből:", missing_teams)

ucl_data.head()

# Kalapok szerint

In [ ]:
# Outcomes by pot

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as font_manager
import os

# ---- STYLE ----
background_color = '#3c3d3d'
text_color = '#5ECB43'
font_path = os.getcwd() + r'\Athletic\Nexa-ExtraLight.ttf'  # ha létezik a font
if os.path.exists(font_path):
    font_props = font_manager.FontProperties(fname=font_path)
else:
    font_props = None  # fallback ha nem található a font

# ---- PREP DATA ----
home_df = ucl_data[['Pot_Home', 'Points_Home']].rename(columns={'Pot_Home': 'Pot', 'Points_Home': 'Points'})
away_df = ucl_data[['Pot_Away', 'Points_Away']].rename(columns={'Pot_Away': 'Pot', 'Points_Away': 'Points'})
combined = pd.concat([home_df, away_df], ignore_index=True)
combined['Result'] = combined['Points'].map({3: 'Win', 1: 'Draw', 0: 'Loss'})

summary = combined.groupby(['Pot', 'Result']).size().unstack(fill_value=0)
summary = summary[['Win', 'Draw', 'Loss']]  # biztosítsuk a sorrendet

# ---- CHART ----
fig, ax = plt.subplots(figsize=(10, 6), facecolor=background_color)
ax.set_facecolor(background_color)

# Színek: Win – Draw – Loss
colors = {
    'Win': '#5ECB43',   # zöld
    'Draw': '#FFD700',  # sárga
    'Loss': '#FF4C4C'   # piros
}

# Egymás melletti (nem stacked) barplot
n_pots = len(summary)
n_results = len(summary.columns)
width = 0.8 / n_results  # kis hézag a potok között
x = np.arange(1, n_pots + 1)

for i, result in enumerate(summary.columns):
    ax.bar(
        x + (i - n_results/2) * width + width/2,
        summary[result],
        width=width,
        label=result,
        color=colors[result]
    )

# Adatfeliratok a barok tetejére
for i, result in enumerate(summary.columns):
    for j, (pot, value) in enumerate(zip(x, summary[result])):
        ax.text(
            pot + (i - n_results/2) * width + width/2,
            value + 0.5,
            str(value),
            ha='center',
            color='white',
            fontsize=10
        )

# ---- FORMATTING ----
ax.set_title('Match Outcomes by Pot', color=text_color, fontsize=20, pad=15)
ax.set_xlabel('Pot', color='white', fontsize=14)
ax.set_ylabel('Number of Matches', color='white', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(summary.index.astype(str), fontsize=15, color='white')
ax.legend(
    facecolor=background_color,
    edgecolor='none',
    labelcolor='white',
    fontsize=11,
    loc='upper right'
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('white')
ax.spines['bottom'].set_color('white')
ax.tick_params(colors='white')
plt.grid(axis='y', color='gray', linestyle='--', linewidth=0.5)

# ---- WATERMARK ----
fig.text(
    0.86, 0.95,
    'ADAM JAKUS',
    color=text_color,
    fontsize=18,
    fontproperties=font_props,
    ha='center',
    va='center'
)

plt.tight_layout()
plt.show()

In [ ]:
# xG by pot boxplot

# Home és Away xG adatok egyesítése
home_xg = ucl_data[['Pot_Home', 'xG_Home']].rename(columns={'Pot_Home': 'Pot', 'xG_Home': 'xG'})
away_xg = ucl_data[['Pot_Away', 'xG_Away']].rename(columns={'Pot_Away': 'Pot', 'xG_Away': 'xG'})
xg_data = pd.concat([home_xg, away_xg], ignore_index=True)

# Boxplot készítése
fig, ax = plt.subplots(figsize=(10, 6), facecolor=background_color)
ax.set_facecolor(background_color)

xg_data['xG'] = pd.to_numeric(xg_data['xG'], errors='coerce')

bp = xg_data.boxplot(
    column='xG', by='Pot', ax=ax, patch_artist=True,
    boxprops=dict(facecolor='#5ECB43', color='white'),
    whiskerprops=dict(color='white'),
    capprops=dict(color='white'),
    medianprops=dict(color='orange', linewidth=2),
    flierprops=dict(marker='o', markerfacecolor='white', alpha=0.5)
)

plt.suptitle('')
ax.set_title('xG Distribution by Pot', color=text_color, fontsize=18, pad=15)
ax.set_xlabel('Pot', color='white')
ax.set_ylabel('xG (Expected Goals)', color='white')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('white')
ax.spines['bottom'].set_color('white')
ax.tick_params(colors='white')
ax.set_xticks(x)
ax.set_xticklabels([str(i) for i in x], fontsize=15)
ax.set_ylim(top=5.1)
plt.grid(axis='y', color='gray', linestyle='--', linewidth=0.5)

# ---- WATERMARK ----
fig.text(
    0.86, 0.925,
    'ADAM JAKUS',
    color=text_color,
    fontsize=18,
    fontproperties=font_props,
    ha='center',
    va='center'
)

plt.tight_layout()
plt.show()


# Erőviszony mátrix

In [ ]:
# ---------- MATRICES (DYNAMIC VERSION) ----------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as font_manager

# ---------- STYLE ----------
background_color = '#3c3d3d'
text_color = '#5ECB43'

df = ucl_data.copy()  # vagy uel_data / uecl_data

# ---------- DATA PREP ----------
df["Goal_Diff_Home"] = df["Goals_Home"] - df["Goals_Away"]
df["Goal_Diff_Away"] = -df["Goal_Diff_Home"]
df["xG_Diff_Home"] = df["xG_Home"] - df["xG_Away"]
df["xG_Diff_Away"] = -df["xG_Diff_Home"]

home_df = df[["Pot_Home", "Pot_Away", "Points_Home", "Goal_Diff_Home", "xG_Diff_Home"]].rename(
    columns={"Pot_Home": "Pot", "Pot_Away": "OppPot", 
             "Points_Home": "Points", "Goal_Diff_Home": "Goal_Diff", "xG_Diff_Home": "xG_Diff"})
away_df = df[["Pot_Away", "Pot_Home", "Points_Away", "Goal_Diff_Away", "xG_Diff_Away"]].rename(
    columns={"Pot_Away": "Pot", "Pot_Home": "OppPot", 
             "Points_Away": "Points", "Goal_Diff_Away": "Goal_Diff", "xG_Diff_Away": "xG_Diff"})
combined = pd.concat([home_df, away_df], ignore_index=True)

combined["Pot"] = combined["Pot"].astype(str)
combined["OppPot"] = combined["OppPot"].astype(str)

# Pot-lista automatikusan
pots = sorted(combined["Pot"].unique(), key=lambda x: int(x))

# ---------- HELPER FUNCTION ----------
def plot_heatmap(matrix, title, cmap, annot_fmt=".2f"):
    fig, ax = plt.subplots(figsize=(7, 6), facecolor=background_color)
    ax.set_facecolor(background_color)

    cax = ax.imshow(matrix, cmap=cmap, interpolation='nearest')

    # Tengelyek
    ax.set_xticks(np.arange(len(matrix.columns)))
    ax.set_yticks(np.arange(len(matrix.index)))
    ax.set_xticklabels(matrix.columns, color='white', fontsize=12)
    ax.set_yticklabels(matrix.index, color='white', fontsize=12)
    plt.setp(ax.get_xticklabels(), rotation=0, ha="center", rotation_mode="anchor")

    # Adatfeliratok
    for i in range(len(matrix.index)):
        for j in range(len(matrix.columns)):
            text = f"{matrix.iloc[i, j]:{annot_fmt}}" if i != j else "-"
            ax.text(j, i, text, ha="center", va="center", color="black", fontsize=10)

    ax.set_title(title, color=text_color, fontsize=16, pad=15)
    ax.spines[:].set_visible(False)
    plt.grid(False)

    # Színtér
    cbar = plt.colorbar(cax, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.yaxis.set_tick_params(color='white')
    plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')

    # Watermark
    fig.text(0.86, -0.02, 'ADAM JAKUS', color=text_color, fontsize=14, ha='center', va='center')

    plt.tight_layout()
    plt.show()

# ---------- 1. Átlagpontok ----------
avg_points = combined.groupby(["Pot", "OppPot"])["Points"].mean().unstack()
avg_points = avg_points.reindex(pots, axis=0).reindex(pots, axis=1)
plot_heatmap(avg_points, "Average Points vs Opponent Pot", cmap="RdYlGn")

# ---------- 2. Átlagos gólkülönbség ----------
avg_goal_diff = combined.groupby(["Pot", "OppPot"])["Goal_Diff"].mean().unstack()
avg_goal_diff = avg_goal_diff.reindex(pots, axis=0).reindex(pots, axis=1)
plot_heatmap(avg_goal_diff, "Average Goal Difference vs Opponent Pot", cmap="RdYlGn")

# ---------- 3. Átlagos xG-különbség ----------
avg_xg_diff = combined.groupby(["Pot", "OppPot"])["xG_Diff"].mean().unstack()
avg_xg_diff = avg_xg_diff.reindex(pots, axis=0).reindex(pots, axis=1)
plot_heatmap(avg_xg_diff, "Average xG Difference vs Opponent Pot", cmap="RdYlGn")


# Meglepetésindex

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------- STYLE ----------
background_color = '#3c3d3d'
text_color = '#5ECB43'

df = ucl_data.copy()

# Pot számokat számként tároljuk (pl. "Pot 1" -> 1)
def extract_pot_num(pot_value):
    if isinstance(pot_value, str):
        return int(''.join([c for c in pot_value if c.isdigit()]))
    return pot_value

df["Pot_Home"] = df["Pot_Home"].apply(extract_pot_num)
df["Pot_Away"] = df["Pot_Away"].apply(extract_pot_num)

# Gólkülönbségek
df["Goal_Diff"] = df["Goals_Home"] - df["Goals_Away"]
df["xG_Diff"] = df["xG_Home"] - df["xG_Away"]

# Pot különbség (pozitív ha a vendég gyengébb potból jön)
df["Pot_Diff"] = df["Pot_Home"] - df["Pot_Away"]

# Surprise Index számítás
df["Surprise_Index"] = (df["Goal_Diff"] - df["xG_Diff"]) * df["Pot_Diff"]

# ---------- TOP & BOTTOM MATCHES ----------
top_surprises = df.sort_values("Surprise_Index", ascending=False).head(10)
bottom_surprises = df.sort_values("Surprise_Index", ascending=True).head(10)

# ---------- VISUALIZATION ----------
fig, ax = plt.subplots(figsize=(10, 6), facecolor=background_color)
ax.set_facecolor(background_color)

ax.barh(top_surprises["Home"] + " vs " + top_surprises["Away"], 
        top_surprises["Surprise_Index"], color="#5ECB43")
ax.barh(bottom_surprises["Home"] + " vs " + bottom_surprises["Away"], 
        bottom_surprises["Surprise_Index"], color="#E74C3C")

ax.set_title("Top 10 Positive & Negative Surprise Matches", color=text_color, fontsize=18, pad=15)
ax.set_xlabel("Surprise Index", color='white')
ax.tick_params(colors='white')
ax.spines[:].set_visible(False)
plt.grid(axis='x', color='gray', linestyle='--', linewidth=0.5)
plt.gca().invert_yaxis()

# Watermark
fig.text(0.86, -0.02, 'ADAM JAKUS', color=text_color, fontsize=14, ha='center', va='center')

plt.tight_layout()
plt.show()

# ---------- SUMMARY BY POT ----------
surprise_by_pot = df.groupby("Pot_Home")["Surprise_Index"].mean()
print(surprise_by_pot)


# Alul- és felülteljesítők

In [ ]:
# ---------- IMPORTS ----------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------- STYLE ----------
background_color = '#3c3d3d'
text_color = '#5ECB43'

df = ucl_data.copy()

# ---------- CALCULATIONS ----------

# Gólkülönbség és xG-különbség
df["Goal_Diff_Home"] = df["Goals_Home"] - df["Goals_Away"]
df["xG_Diff_Home"] = df["xG_Home"] - df["xG_Away"]
df["Goal_Diff_Away"] = -df["Goal_Diff_Home"]
df["xG_Diff_Away"] = -df["xG_Diff_Home"]

# Egyesített adat (minden meccs két irányból)
home_df = df[["Home", "Pot_Home", "Goal_Diff_Home", "xG_Diff_Home"]].rename(
    columns={"Home": "Team", "Pot_Home": "Pot", "Goal_Diff_Home": "Goal_Diff", "xG_Diff_Home": "xG_Diff"})
away_df = df[["Away", "Pot_Away", "Goal_Diff_Away", "xG_Diff_Away"]].rename(
    columns={"Away": "Team", "Pot_Away": "Pot", "Goal_Diff_Away": "Goal_Diff", "xG_Diff_Away": "xG_Diff"})
combined = pd.concat([home_df, away_df], ignore_index=True)

# Overperformance (xG-hez képest)
combined["Overperformance"] = combined["Goal_Diff"] - combined["xG_Diff"]

# ---------- AGGREGÁLT EREDMÉNYEK ----------
team_perf = (
    combined.groupby(["Team", "Pot"])["Overperformance"]
    .mean()
    .reset_index()
    .sort_values("Overperformance", ascending=False)
)

# Top és bottom csapatok
top_5 = team_perf.head(5)
bottom_5 = team_perf.tail(5)

# ---------- PLOT ----------
fig, ax = plt.subplots(figsize=(9,6), facecolor=background_color)
ax.set_facecolor(background_color)

bars_top = ax.barh(top_5["Team"], top_5["Overperformance"], color="#5ECB43", label="Overperformers")
bars_bottom = ax.barh(bottom_5["Team"], bottom_5["Overperformance"], color="#D64E4E", label="Underperformers")

# ---------- STYLE FIXES ----------
ax.spines['top'].set_color('white')
ax.spines['bottom'].set_color('white')
ax.spines['left'].set_color('white')
ax.spines['right'].set_color('white')
ax.tick_params(colors='white')  # tengely tickek
ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')

# ---------- TITLE & AXES ----------
ax.set_title("Top & Bottom xG Over/Underperformers", color=text_color, fontsize=18, pad=15)
ax.set_xlabel("Goal Diff – xG Diff (avg per match)", color='white', fontsize=12)
ax.axvline(0, color='white', linewidth=1)

# Annotációk
for bars in [bars_top, bars_bottom]:
    for bar in bars:
        ax.text(bar.get_width() + (0.05 if bar.get_width() > 0 else -0.05),
                bar.get_y() + bar.get_height()/2,
                f"{bar.get_width():.2f}",
                va='center', ha='left' if bar.get_width() > 0 else 'right',
                color='white', fontsize=10)

# Legenda és watermark
ax.legend(facecolor=background_color, edgecolor="none", labelcolor='white', fontsize=10)
fig.text(0.86, -0.02, 'ADAM JAKUS', color=text_color, fontsize=14, ha='center', va='center')

plt.tight_layout()
plt.show()


# Expected Points

In [ ]:
# ---------- IMPORTS ----------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson
from adjustText import adjust_text  # <-- automatikus feliratigazítás

# ---------- STYLE ----------
background_color = '#3c3d3d'
text_color = '#5ECB43'

# ---------- DATA ----------
df = ucl_data.copy()

# ---------- SEGÉDFÜGGVÉNY ----------
def expected_points(xg_for, xg_against):
    """Poisson-eloszlás alapján számolja a várható pontokat."""
    goals = np.arange(0, 10)
    p_for = poisson.pmf(goals, xg_for)
    p_against = poisson.pmf(goals, xg_against)
    match_matrix = np.outer(p_for, p_against)
    p_win = np.tril(match_matrix, -1).sum()
    p_draw = np.trace(match_matrix)
    return 3 * p_win + 1 * p_draw

# ---------- SZÁMÍTÁSOK ----------
df["xPoints_Home"] = df.apply(lambda x: expected_points(x["xG_Home"], x["xG_Away"]), axis=1)
df["xPoints_Away"] = df.apply(lambda x: expected_points(x["xG_Away"], x["xG_Home"]), axis=1)

home_df = df[["Home", "Pot_Home", "Points_Home", "xPoints_Home"]].rename(
    columns={"Home": "Team", "Pot_Home": "Pot", "Points_Home": "Points", "xPoints_Home": "xPoints"})
away_df = df[["Away", "Pot_Away", "Points_Away", "xPoints_Away"]].rename(
    columns={"Away": "Team", "Pot_Away": "Pot", "Points_Away": "Points", "xPoints_Away": "xPoints"})
combined = pd.concat([home_df, away_df], ignore_index=True)

# ---------- AGGREGÁLÁS ----------
team_perf = (
    combined.groupby(["Team", "Pot"])[["Points", "xPoints"]]
    .sum()
    .reset_index()
)
team_perf["Diff"] = team_perf["Points"] - team_perf["xPoints"]

# ---------- PLOT ----------
fig, ax = plt.subplots(figsize=(9, 7), facecolor=background_color)
ax.set_facecolor(background_color)

# Színezés: zöld, ha túlteljesített, piros, ha alul
colors = team_perf["Diff"].apply(lambda x: "#5ECB43" if x > 0 else "#D64E4E")

ax.scatter(team_perf["xPoints"], team_perf["Points"],
           s=80, color=colors, edgecolor="white", linewidth=0.8, alpha=0.9)

# Átló
min_val = min(team_perf["xPoints"].min(), team_perf["Points"].min())
max_val = max(team_perf["xPoints"].max(), team_perf["Points"].max())
ax.plot([min_val, max_val], [min_val, max_val], color="gray", linestyle="--", linewidth=1)

# Feliratok automatikus igazítással
texts = []
for _, row in team_perf.iterrows():
    label = f"{row['Team']} ({row['Diff']:+.1f})"
    texts.append(ax.text(row["xPoints"], row["Points"], label,
                         color="white", fontsize=9, ha='left', va='center'))
adjust_text(texts, arrowprops=dict(arrowstyle='-', color='white', lw=0.5))

# Tengelyek és cím
ax.set_xlabel("Expected Points", color='white', fontsize=12)
ax.set_ylabel("Actual Points", color='white', fontsize=12)
ax.set_title("Actual vs Expected Points", color=text_color, fontsize=18, pad=15)

# Tickek, színek, keretek
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('white')

# Watermark
fig.text(0.86, -0.02, 'ADAM JAKUS', color=text_color, fontsize=14, ha='center', va='center')

plt.tight_layout()
plt.show()
